# Vintern GGUF Invoice OCR Local Pipeline

Notebook này là pipeline tự chứa: input là ảnh hoặc PDF, PDF sẽ được render thành ảnh, model `rootonchair/Vintern-1B-v3_5-GGUF-ext` đọc hóa đơn qua `llama.cpp server`, output là file JSON theo cấu trúc demo.

Vintern là vision-language model nên `llama-server` cần cả model GGUF và file `mmproj`. Notebook sẽ tự tải `mmproj`, sau đó in ra command server để bạn copy sang terminal riêng.

```powershell
llama-server -hf rootonchair/Vintern-1B-v3_5-GGUF-ext:Q4_K_M --mmproj "D:/Invoice_Scanner/models/mmproj-Vintern-1B-v3_5-Q8_0.gguf" --chat-template vicuna --n-gpu-layers 999 --port 8081
```

Nếu muốn nhẹ hơn có thể đổi `Q4_K_M` thành `Q4_K_S` hoặc `IQ4_XS`.

In [ ]:
import sys

!{sys.executable} -m pip install --upgrade pillow pymupdf requests huggingface_hub

In [ ]:
import ast
import base64
import json
import re
import subprocess
from io import BytesIO
from pathlib import Path

import pymupdf as fitz
import requests
from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

## Cấu hình input/output

Chỉ cần đổi `INPUT_PATH`. Có thể là `.pdf`, `.jpg`, `.png`, `.webp`, `.tif`, ...

In [ ]:
INPUT_PATH = Path("D:/Invoice_Scanner/invoice.pdf")

SERVER_URL = "http://127.0.0.1:8081/v1"
SERVER_MODEL = "local-model"

MODEL_REPO_ID = "rootonchair/Vintern-1B-v3_5-GGUF-ext"
MODEL_QUANT = "Q4_K_M"
MMPROJ_REPO_ID = "ngxson/Vintern-1B-v3_5-GGUF"
MMPROJ_FILENAME = "mmproj-Vintern-1B-v3_5-Q8_0.gguf"
MODEL_DIR = Path("D:/Invoice_Scanner/models")
MMPROJ_PATH = MODEL_DIR / MMPROJ_FILENAME

PDF_DPI = 300
MAX_IMAGE_SIDE = 2600
JPEG_QUALITY = 95
MAX_TOKENS = 2048
TEMPERATURE = 0.0
USE_IMAGE_ENHANCEMENT = True
USE_PDF_TEXT_LAYER = True
PDF_TEXT_STRATEGY = "assist"  # assist: chạy model rồi bù text layer; off: chỉ model; fast: ưu tiên text layer
USE_GPU_IF_AVAILABLE = True
GPU_LAYERS = 999
INCLUDE_RAW_RESPONSE_IN_RESULTS = False

OUTPUT_JSON = INPUT_PATH.with_suffix(".vintern_gguf.json")
PDF_IMAGE_DIR = INPUT_PATH.with_name(f"{INPUT_PATH.stem}_images")

print("Input:", INPUT_PATH)
print("Output JSON:", OUTPUT_JSON)
print("Server:", SERVER_URL)
print("PDF text strategy:", PDF_TEXT_STRATEGY)
print("GPU auto:", USE_GPU_IF_AVAILABLE)
print("MMProj:", MMPROJ_PATH)

## Tải mmproj và mở server

Chạy cell này trước, copy command được in ra sang một terminal PowerShell riêng, rồi giữ terminal server đó mở.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

downloaded_mmproj = Path(hf_hub_download(
    repo_id=MMPROJ_REPO_ID,
    filename=MMPROJ_FILENAME,
    local_dir=MODEL_DIR,
))

MMPROJ_PATH = downloaded_mmproj.resolve()

if not MMPROJ_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy mmproj sau khi tải: {MMPROJ_PATH}")

def has_nvidia_gpu():
    if not USE_GPU_IF_AVAILABLE:
        return False
    try:
        result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=5)
        return result.returncode == 0 and bool(result.stdout.strip())
    except (FileNotFoundError, subprocess.SubprocessError):
        return False


n_gpu_layers = GPU_LAYERS if has_nvidia_gpu() else 0

server_command = (
    f'llama-server -hf {MODEL_REPO_ID}:{MODEL_QUANT} '
    f'--mmproj "{MMPROJ_PATH}" '
    f'--chat-template vicuna '
    f'--n-gpu-layers {n_gpu_layers} '
    f'--port 8081'
)

print("Đã có mmproj:", MMPROJ_PATH)
print("GPU layers:", n_gpu_layers)
print("Kiểm tra PowerShell:", f'Test-Path "{MMPROJ_PATH}"')
print("\nCopy lệnh này sang terminal PowerShell riêng và giữ terminal đó mở:\n")
print(server_command)

In [ ]:
# Schema/prompt đang dùng thật trong backend pipeline.
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path("D:/Invoice_Scanner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.pipeline import INVOICE_JSON_SCHEMA, INVOICE_PROMPT

print(json.dumps(INVOICE_JSON_SCHEMA, ensure_ascii=False, indent=2))


In [ ]:
# Dùng chung backend pipeline đã được sửa để notebook và web app không lệch logic.
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path("D:/Invoice_Scanner")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.app.pipeline import (
    OCRConfig,
    build_llama_server_command,
    get_server_media_marker,
    load_input_images,
    probe_server as probe_backend_server,
    scan_invoice_file,
)


def make_ocr_config():
    return OCRConfig(
        server_url=SERVER_URL,
        model=SERVER_MODEL,
        pdf_dpi=PDF_DPI,
        max_image_side=MAX_IMAGE_SIDE,
        jpeg_quality=JPEG_QUALITY,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        request_timeout=600,
        use_image_enhancement=USE_IMAGE_ENHANCEMENT,
        use_pdf_text_layer=USE_PDF_TEXT_LAYER,
        pdf_text_strategy=PDF_TEXT_STRATEGY,
        include_raw_response=INCLUDE_RAW_RESPONSE_IN_RESULTS,
    )


def probe_server(server_url=SERVER_URL, timeout=10):
    return probe_backend_server(server_url, timeout=timeout)["checks"]


def save_results(results, output_path, include_raw_response=False):
    output = []
    for result in results:
        row = dict(result)
        if not include_raw_response:
            row.pop("raw_response", None)
        output.append(row)

    output_path = Path(output_path)
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


In [ ]:
# Pipeline OCR cũ trong notebook đã được thay bằng backend.app.pipeline.
# Giữ cell này để notebook không còn định nghĩa lại parser cũ làm mất cột số tiền/VAT.


## Kiểm tra server

In [ ]:
for item in probe_server(SERVER_URL):
    print(item)

print("Media marker server:", get_server_media_marker(SERVER_URL))

## Xem trước input

In [ ]:
preview_images, preview_source_type, preview_image_paths = load_input_images(
    INPUT_PATH,
    pdf_dpi=PDF_DPI,
    save_pdf_images=INPUT_PATH.suffix.lower() == ".pdf",
    pdf_image_dir=PDF_IMAGE_DIR,
)

print("Loại file:", preview_source_type)
print("Số trang/ảnh:", len(preview_images))
if any(preview_image_paths):
    print("Ảnh đã lưu:", preview_image_paths[0])

display(preview_images[0])

## OCR và lưu JSON

In [ ]:
ocr_config = make_ocr_config()
results = scan_invoice_file(
    INPUT_PATH,
    output_image_dir=PDF_IMAGE_DIR,
    config=ocr_config,
)
results


In [ ]:
save_results(results, OUTPUT_JSON, include_raw_response=INCLUDE_RAW_RESPONSE_IN_RESULTS)
print("Đã lưu:", OUTPUT_JSON)

if results:
    first_items = results[0].get("data", {}).get("items", [])
    print("Dòng hàng hóa/dịch vụ:")
    print(json.dumps(first_items, ensure_ascii=False, indent=2))
